### Query Translation
Query translation sits at the first stage of an advanced RAG pipeline. The goal of query translation is to take the input user question and to translate it in some way as to improve retrival.

Semantic search on embeddings is hard to get right. Embedding long documents is especially challenging. User queries are a challenge too. If the user provides an ambigious query, they'll end up get an ambiguous matches from embeddings and consequently an ambguous answer. The ambiguous matches land up in the LLM's context from which comes the LLM's response, which could lead to hallucinations.

There are 2 broad approaches to tackle the above issue:
1. Multi Query
2. RAG Fusion

#### Multi-Query

**What is it?**

You take the user’s query and generate multiple (N=5?) _paraphrases/reformulations_ of it (using an LLM). Each paraphrase is run against the retriever/vector DB _independently_, and the results are merged (i.e. we pick unique values across N queries). These results are then passed to the LLM as the _context_ from which the LLM generates its response. The intuition is that by asking the LLM the same question in N different ways, we will get more relevents chunks of data into the context, thereby improving overall response.

**Why use it?**

Different phrasings capture different embeddings → retrieve more relevant chunks. Helps reduce “embedding mismatch” (e.g., synonyms, domain-specific terms).

**Example:**

User asks: _"How do I cook pasta quickly?"_
LLM generates (3 variations in this case):
* "fast ways to prepare pasta"
* "quick pasta cooking methods"
* "rapid spaghetti preparation"

All run → retrieve docs covering microwaving, pressure cooker, etc. (there could be duplicates, so generate a unique list of retrievals). The retriever might otherwise miss some if only the original query was used.

**Key point:** Multi-query improves recall by broadening query formulations.

The diagram below illustrates this technique.

![Multi Query](../images/multi_query.png)

In [1]:
import bs4, os
import pathlib
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown

from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# since we are using Gemini, we'll use Google embeddings
# from langchain_google_genai import GoogleGenerativeAIEmbeddings
# I seem to have permanently exhausted rate limt on Google embeddings on free tier,
# I don't want to enable billing, so am switching to Cohere embeddings
from langchain_cohere import CohereEmbeddings
from langchain_community.vectorstores import FAISS

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
# load API keys from .env files
load_dotenv(override=True)

# for colorful text output
# when using Console in a Notebook editor, especially in VS code, use the following
# alternate instantiation, rather than the default console = Console()

# console = Console()
console = Console(force_jupyter=False, force_terminal=True, width=120, soft_wrap=True)

In [6]:
# create our LLM - we'll be using Gemini-2.5-flash
llm = init_chat_model("google_genai:gemini-2.5-flash", temperature=0.0)
embeddings = CohereEmbeddings(model="embed-english-v3.0")
faiss_store = pathlib.Path(os.getcwd()) / "../faiss_index_rag_mq"

In [7]:
def create_or_load_embeddings():
    """creates (if not available) or loads from disk a FAISS embedding"""
    if not faiss_store.exists():
        # in this example we'll load document from a URL
        web_url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
        console.print(
            f"[yellow]Loading document from URL {web_url}. Please wait...[/yellow]"
        )
        loader = WebBaseLoader(
            web_paths=(web_url,),
            bs_kwargs=dict(
                parse_only=bs4.SoupStrainer(
                    class_=("post-content", "post-title", "post-header")
                )
            ),
        )
        blog_docs = loader.load()

        console.print(f"[blue]Loaded {len(blog_docs)} documents from URL[/blue]")
        console.print(
            f"[blue]Metadata of first document: {blog_docs[0].metadata}[/blue]"
        )
        console.print(
            f"[blue]First 200 chars of first document: {blog_docs[0].page_content[:200]}[/blue]"
        )

        # split document into chunks of 1000 chars with 200 chars overlap
        console.print(f"[yellow]Chunking the PDF. Please wait...[/yellow]")

        text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
            chunk_size=300, chunk_overlap=50
        )

        # Make splits
        splits = text_splitter.split_documents(blog_docs)
        console.print(f"[blue]Created {len(splits)} chunks[/blue]")

        # save to embeddings
        console.print("[yellow]Creating embeddings. Please wait...[/yellow]")
        # Use a Gemini embedding model that is suitable for retrieval.
        # It is important to match the model to the task.
        # embeddings = GoogleGenerativeAIEmbeddings(
        #     model="models/text-embedding-004",
        #     task_type="retrieval_document",
        # )
        vector_store = FAISS.from_documents(documents=splits, embedding=embeddings)
        retriever = vector_store.as_retriever()
        vector_store.save_local(str(faiss_store))
        console.print(
            f"[yellow]Local embeddings created at {str(faiss_store)}[/yellow]"
        )
    else:
        console.print(
            f"[yellow]Loading existing embeddings from {str(faiss_store)}[/yellow]"
        )
        # embeddings = GoogleGenerativeAIEmbeddings(
        #     model="models/text-embedding-004",
        #     task_type="retrieval_document",
        # )
        vector_store = FAISS.load_local(
            str(faiss_store), embeddings, allow_dangerous_deserialization=True
        )
        retriever = vector_store.as_retriever()

    return retriever

In [8]:
retriever = create_or_load_embeddings()

Loading document from URL https://lilianweng.github.io/posts/2023-06-23-agent/. Please wait...
Loaded 1 documents from URL
Metadata of first document: {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}
First 200 chars of first document: 

      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a 
Chunking the PDF. Please wait...
Created 50 chunks
Creating embeddings. Please wait...
Local embeddings created at /home/mjbhobe/code/git-projects/learning_langchain/src/langchain_tutorial/Advanced RAG/../faiss_index_rag_mq


In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# from langchain_google_genai import ChatGoogleGenerativeAI

# Multi Query: Different Perspectives
template = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines. 

Original question: {question}"""

prompt_perspectives = ChatPromptTemplate.from_template(template)

In [10]:
generate_queries = (
    prompt_perspectives | llm | StrOutputParser() | (lambda x: x.split("\n"))
)

# let's try invoking the chain
generate_queries.invoke({"question": "What is task decomposition for LLM agents?"})

['Define task decomposition for large language model agents.',
 'Why do LLM agents employ task decomposition?',
 'How is task decomposition implemented or performed by LLM agents?',
 'What are the advantages of using task decomposition in LLM agent systems?',
 'Describe the process of breaking down complex problems for AI agents utilizing large language models.']

So you notice that we generated 5 different versions of the same query to improve our matches against the vector database.

In [11]:
from langchain.load import dumps, loads


def get_unique_union(documents: list[list]):
    """Unique union of retrieved docs"""
    # Flatten list of lists, and convert each Document to string
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # Get unique documents
    unique_docs = list(set(flattened_docs))
    # Return
    return [loads(doc) for doc in unique_docs]

In [12]:
# Retrieve
question = "What is task decomposition for LLM agents?"
# here we are firing multiple queries against the vector store, getting all the
# responses & creating a unique set from all the responses.
retrieval_chain = generate_queries | retriever.map() | get_unique_union
docs = retrieval_chain.invoke({"question": question})
print(f"Got {len(docs)} documents")
for i, doc in enumerate(docs):
    console.print(
        Markdown(f"### Document {i+1}\n{doc.page_content[:50] + "..."}\n---\n")
    )

# and print the retrival chain too
console.print(f"Retrieval chain: {retrieval_chain}")

Got 8 documents
                                                       Document 1                                                       


                                 (2) Model selection: LLM distributes the tasks to ...                                  
                                                       Document 2                                                       


                                 [17] Bran et al. “ChemCrow: Augmenting large-langu...                                  
                                                       Document 3                                                       


                                 [4] Liu et al. “LLM+P: Empowering Large Language M...                                  
                                                       Document 4                                                       


                                 Component One: Planning# A complicated task usuall...                                  
        

/tmp/ipykernel_9303/3928205432.py:11: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc) for doc in unique_docs]


In [13]:
from operator import itemgetter

# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

final_rag_chain = (
    {"context": retrieval_chain, "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

response = final_rag_chain.invoke({"question": question})
console.print(Markdown(response))

Task decomposition for LLM agents is the process by which a large or complicated task is broken down into smaller, more manageable subgoals or steps. This enables the agent to handle complex tasks efficiently and enhances its performance.

Methods for task decomposition include:

 • Chain of Thought (CoT): Instructing the LLM to "think step by step" to break down hard tasks into simpler steps.     
 • Tree of Thoughts: Extending CoT by exploring multiple reasoning possibilities at each step, generating multiple thoug
 • Simple Prompting: Using prompts like "Steps for XYZ." or "What are the subgoals for achieving XYZ?".                 
 • Task-specific instructions: Providing specific guidance, e.g., "Write a story outline."                              
 • Human inputs: Incorporating human guidance for decomposition.                                                        
 • LLM as a parser: The LLM acts as the "brain" to parse user requests into multiple tasks, each with attributes l